# **1. Introduction**<br>
Automatic number plate recognition (ANPR) is a crucial technology for intelligent transportation systems, law enforcement, and traffic management. However, most ANPR systems are designed for Latin scripts and do not work well for Bengali number plates, which are widely used in Bangladesh.
This notebook demonstrates a complete pipeline for detecting and recognizing Bengali vehicle number plates using deep learning and OCR techniques.

**PARSING DATA FROM XML AND CONVERTING IT INTO CSV**

In [ ]:
import pandas as pd
import xml.etree.ElementTree as xet

In [ ]:
from glob import glob

path = glob('/kaggle/input/bangladeshi-vehicle-number-plates/number_Plate_raw/*.xml')

In [ ]:
path = glob('/kaggle/input/bangladeshi-vehicle-number-plates/train_image/train_image/*.xml')

In [ ]:
labels_dict = dict(filepath=[],xmin=[],xmax=[],ymin=[],ymax=[])
for filename in path:

    info = xet.parse(filename)
    root = info.getroot()
    member_object = root.find('object')
    labels_info = member_object.find('bndbox')
    xmin = int(labels_info.find('xmin').text)
    xmax = int(labels_info.find('xmax').text)
    ymin = int(labels_info.find('ymin').text)
    ymax = int(labels_info.find('ymax').text)

    labels_dict['filepath'].append(filename)
    labels_dict['xmin'].append(xmin)
    labels_dict['xmax'].append(xmax)
    labels_dict['ymin'].append(ymin)
    labels_dict['ymax'].append(ymax)

In [ ]:
df = pd.DataFrame(labels_dict)
df.to_csv('labels.csv',index=False)
df.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2

**READ THE IMAGE FILE**

In [ ]:
filename = df['filepath'][0]
def getFilename(filename):
    filename_image = xet.parse(filename).getroot().find('filename').text
    filepath_image = os.path.join('/kaggle/input/bangladeshi-vehicle-number-plates/train_image/train_image',filename_image)
    return filepath_image
getFilename(filename)

In [ ]:
image_path = list(df['filepath'].apply(getFilename))
image_path[:10]#random check

**VERIFY THE DATA**

file_path = image_path[3] #path of our image 
img = cv2.imread(file_path) #read the image
# 1198	1509	2905	3131 
img = io.imread(file_path) #Read the image
fig = px.imshow(img)
fig.update_layout(width=600, height=500, margin=dict(l=10, r=10, b=10, t=10),xaxis_title='Figure 1 - image with bounding box')
fig.add_shape(type='rect',x0=1198, x1=1509, y0=2905, y1=3131, xref='x', yref='y',line_color='cyan')

**DATA PROCESSING**

i. read data

In [ ]:
!pip install --upgrade tensorflow-io

In [ ]:
!pip install --upgrade scipy

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:
#Targeting all our values in array selecting all columns
labels = df.iloc[:,1:].values
data = []
output = []
for ind in range(len(image_path)):
    image = image_path[ind]
    img_arr = cv2.imread(image)
    h,w,d = img_arr.shape
    # Prepprocesing
    load_image = load_img(image,target_size=(224,224))
    load_image_arr = img_to_array(load_image)
    norm_load_image_arr = load_image_arr/255.0 # Normalization
    # Normalization to labels
    xmin,xmax,ymin,ymax = labels[ind]
    nxmin,nxmax = xmin/w,xmax/w
    nymin,nymax = ymin/h,ymax/h
    label_norm = (nxmin,nxmax,nymin,nymax) # Normalized output
    # Append
    data.append(norm_load_image_arr)
    output.append(label_norm)

In [ ]:
data

**ii. SPLIT ,Train & TEST SET**

In [ ]:
# Convert data to array
X = np.array(data,dtype=np.float32)
y = np.array(output,dtype=np.float32)

In [ ]:
X.shape,y.shape

In [ ]:
# Split the data into training and testing set using sklearn.
x_train,x_test,y_train,y_test = train_test_split(X,y,train_size=0.9,random_state=0)
x_train.shape,x_test.shape,y_train.shape,y_test.shape

# **DEEP LEARNING FOR OBJECT DETECTION**

In [ ]:
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.layers import Dense,Dropout,Flatten,Input
from tensorflow.keras.models import Model
import tensorflow as tf

In [ ]:
inception_resnet = InceptionResNetV2(weights="imagenet",include_top=False, input_tensor=Input(shape=(224,224,3)))
# ---------------------
headmodel = inception_resnet.output
headmodel = Flatten()(headmodel)
headmodel = Dense(500,activation="relu")(headmodel)
headmodel = Dense(250,activation="relu")(headmodel)
headmodel = Dense(4,activation='sigmoid')(headmodel)


# ---------- model
model = Model(inputs=inception_resnet.input,outputs=headmodel)

In [ ]:
# Complie model
model.compile(loss='mse',optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))
model.summary()

**INCEPTION-RESNET-V2 TRAINING AND SAVE**

In [ ]:
!pip install --upgrade tensorflow

In [ ]:
!pip uninstall keras -y

!pip install --upgrade cupy-cuda11x
!pip install --upgrade dill
!pip install --upgrade pyarrow
!pip install --upgrade numpy
!pip install --upgrade protobuf
!pip install --upgrade dask
!pip install --upgrade scipy
!pip install --upgrade tensorflow

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

In [ ]:
# Create a TensorBoard callback
log_dir = "/kaggle/working/logs"  # Change this to your preferred log directory
tensorboard_callback = TensorBoard(log_dir=log_dir)

In [ ]:
tfb = TensorBoard('object_detection')
history = model.fit(x=x_train,y=y_train,batch_size=10,epochs=300,
                    validation_data=(x_test,y_test),callbacks=[tensorboard_callback])

In [ ]:
tfb = TensorBoard('object_detection')
history = model.fit(x=x_train,y=y_train,batch_size=10,epochs=320,
                    validation_data=(x_test,y_test),callbacks=[tensorboard_callback],initial_epoch=310)

In [ ]:
# Get the path of the current directory
output_dir = os.getcwd()

# Create a new directory to store the model
model_dir = os.path.join(output_dir, 'model')
os.makedirs(model_dir, exist_ok=True)

In [ ]:
#model save
model.save('/kaggle/working/model/object_detectionv3.h5')

**TensorBoard**

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir="/kaggle/working/logs"

In [ ]:
!tensorboard --logdir="/kaggle/working/logs"

In [ ]:
import os
import numpy as np
import cv2
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:
#load Model
model = tf.keras.models.load_model('/kaggle/working/model/object_detectionv3.h5')
print('model loaded sucessfully')

In [ ]:
path = '/kaggle/input/test-image/test3.jpg'
def object_detection(path):
    
    # Read image
    image = load_img(path) # PIL object
    image = np.array(image,dtype=np.uint8) # 8 bit array (0,255)
    image1 = load_img(path,target_size=(224,224))
    
    # Data preprocessing
    image_arr_224 = img_to_array(image1)/255.0 # Convert to array & normalized
    h,w,d = image.shape
    test_arr = image_arr_224.reshape(1,224,224,3)
    
    # Make predictions
    coords = model.predict(test_arr)
    
    # Denormalize the values
    denorm = np.array([w,w,h,h])
    coords = coords * denorm
    coords = coords.astype(np.int32)
    
    # Draw bounding on top the image
    xmin, xmax,ymin,ymax = coords[0]
    pt1 =(xmin,ymin)
    pt2 =(xmax,ymax)
    print(pt1, pt2)
    cv2.rectangle(image,pt1,pt2,(0,255,0),5)
    return image, coords

image, cods = object_detection(path)

fig = px.imshow(image)
fig.update_layout(width=700, height=500, margin=dict(l=10, r=10, b=10, t=10),xaxis_title='Figure - Plate Detection')

In [ ]:
(1284, 2577) (1826, 2854)